In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 1. Scaled Dot-Product Attention
# q: [B, H, T, Dh], k/v: [B, H, S, Dh]
def scaled_dot_product_attention(q, k, v, mask=None):
    assert q.size(-1) == k.size(-1), "Q/K 的 head_dim 必须相同"
    assert k.size(-2) == v.size(-2), "K/V 的序列长度必须相同"
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [B,H,T,S]
    if mask is not None:
        assert torch.broadcast_shapes(scores.shape, mask.shape) == scores.shape
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    attn = torch.softmax(scores, dim=-1)  # 沿 key 维 S 做 softmax
    return attn @ v, attn

B, H, T, S, Dh = 2, 4, 5, 6, 8
q = torch.randn(B, H, T, Dh, device=device)
k = torch.randn(B, H, S, Dh, device=device)
v = torch.randn(B, H, S, Dh, device=device)
out, attn = scaled_dot_product_attention(q, k, v)
assert out.shape == (B, H, T, Dh)
assert torch.allclose(attn.sum(-1), torch.ones(B, H, T, device=device), atol=1e-5)

In [ ]:
# 2. Causal Multi-Head Self-Attention
class CausalMHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()[None, None]
        if padding_mask is not None:
            assert padding_mask.shape == (B, T)  # True 表示有效 token
            mask = mask & padding_mask[:, None, None, :]
        context, attn = scaled_dot_product_attention(q, k, v, mask)
        context = context.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(self.dropout(context)), attn

x = torch.randn(2, 5, 32, device=device)
mha = CausalMHA(32, 4).to(device)
y, attn = mha(x)
assert y.shape == x.shape
assert torch.count_nonzero(attn.triu(diagonal=1)) == 0